In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import os, warnings 
warnings.filterwarnings('ignore') 

from setup import *
from glob import glob
import stan

## Loading the dataset

In [2]:
sample_list = []
for x in glob('data/Ren_processed/*'):
    sample_list.append(x.split(r'/')[-1][:-5])
sample_list.sort()

In [3]:
adatas = dict()
mats_tfa = dict()
mats_pwa = dict() 
mats_d2c = dict() 

for sample in sample_list:
    adatas[sample] = sc.read_h5ad("data/Ren_processed/{}.h5ad".format(sample))
    sc.pp.normalize_total(adatas[sample])
    sc.pp.log1p(adatas[sample])
    
    adata = sc.read_h5ad("results/ren_tfa/{}.h5ad".format(sample))
    mats_tfa[sample] = stan.get_activity(adata, key='tfa_stan').to_df()
    
    adata = sc.read_h5ad("results/ren_pwa/{}.h5ad".format(sample))
    mats_pwa[sample] = stan.get_activity(adata, key='pwa_stan').to_df()
    
    mats_d2c[sample] = sc.read_h5ad("results/ren_d2c/{}.h5ad".format(sample)).to_df()

In [4]:
sample_list = []
for x in glob('data/Sonpatki_processed//*'):
    sample_list.append(x.split(r'/')[-1][:-5])
sample_list.sort()

In [5]:
for sample in sample_list:
    adatas[sample] = sc.read_h5ad("data/Sonpatki_processed/{}.h5ad".format(sample))
    sc.pp.normalize_total(adatas[sample])
    sc.pp.log1p(adatas[sample])
    
    adata = sc.read_h5ad("results/sonpatki_tfa/{}.h5ad".format(sample))
    mats_tfa[sample] = stan.get_activity(adata, key='tfa_stan').to_df()
    
    adata = sc.read_h5ad("results/sonpatki_pwa/{}.h5ad".format(sample))
    mats_pwa[sample] = stan.get_activity(adata, key='pwa_stan').to_df()
    
    mats_d2c[sample] = sc.read_h5ad("results/sonpatki_d2c/{}.h5ad".format(sample)).to_df()

In [7]:
sample_list = adatas.keys()
sample_list

dict_keys(['Ren2', 'Ren3', 'Ren51', 'SNU16B', 'SNU18A', 'SNU18Afrozen', 'SNU18B', 'SNU21Afrozen', 'SNU21B', 'SNU21Dfrozen', 'SNU24A', 'SNU24B', 'SNU25A', 'SNU25B', 'SNU27A', 'SNU27B', 'SNU43A', 'SNU51A', 'SNU51B'])

## Preparation for correlation

In [8]:
df_lr_pair = stan.load_lr_pair(source_dir="resources/")
ligand_list = list(set(df_lr_pair['ligand_gene_symbol']))
receptor_list = list(set(df_lr_pair['receptor_gene_symbol']))
pathway_list = mats_pwa[sample].columns.to_list()

In [9]:
tf_list = pd.Index([])
for sample in sample_list:
    tf_list = tf_list.union(mats_tfa[sample].columns)
tf_list = tf_list.to_list()

In [10]:
drug_list = pd.Index([])
for sample in sample_list:
    drug_list = drug_list.union(mats_d2c[sample].columns)
drug_list = list(set(drug_list))

In [17]:
with open('dataframes/ravi_selected_genes.txt', 'r') as f:
    gene_set0 = {line.strip() for line in f if line.strip()}
with open('dataframes/ren_selected_genes.txt', 'r') as f:
    gene_set1 = {line.strip() for line in f if line.strip()}
with open('dataframes/sonpatki_selected_genes.txt', 'r') as f:
    gene_set2 = {line.strip() for line in f if line.strip()}

In [18]:
gene_list = sorted(
    ((gene_set0 & gene_set1) | (gene_set0 & gene_set2)) - set(ligand_list) - set(receptor_list)
)
len(gene_list)

8169

In [19]:
mat_neighbor = dict()
for sample in sample_list:
    mat_neighbor[sample] = stan.compute_spatial_expression(adatas[sample], list(adatas[sample].var.index))

In [20]:
path = f'dataframes/correlation_ren'
if not os.path.exists(path):
    os.mkdir(path)
features = ['ligand', 'receptor', 'gene', 'drug', 'tf', 'pathway']
for f1 in features:
    for f2 in features:
        path = f'dataframes/correlation_ren/{f1}_with_{f2}'
        if not os.path.exists(path):
            os.mkdir(path)

## Correlation with drugs

In [21]:
for drug in drug_list[:3]: ## remove [:3]
    df_corr_drug = pd.DataFrame(np.nan, index=drug_list, columns = sample_list)
    df_corr_gene = pd.DataFrame(np.nan, index=gene_list, columns = sample_list)
    df_corr_liga = pd.DataFrame(np.nan, index=ligand_list, columns = sample_list)
    df_corr_rece = pd.DataFrame(np.nan, index=receptor_list, columns = sample_list)
    df_corr_path = pd.DataFrame(np.nan, index=pathway_list, columns = sample_list)
    df_corr_tf = pd.DataFrame(np.nan, index=tf_list, columns = sample_list)
    
    for sample in sample_list:
        adata = adatas[sample]
        if drug in mats_d2c[sample]:
            drugs = np.setdiff1d(mats_d2c[sample].columns, [drug]).tolist()
            mat_dg = mats_d2c[sample][drug]
            
            ligands = np.intersect1d(adata.var_names, ligand_list).tolist()
            receptors = np.intersect1d(adata.var_names, receptor_list).tolist()
            genes = np.intersect1d(adata.var_names, gene_list[:3]).tolist() ## remove [:3]
            tfs = mats_tfa[sample].columns.to_list()
            
            mat_gene = mat_neighbor[sample][genes]
            mat_ligand = mat_neighbor[sample][ligands]
            mat_receptor = mat_neighbor[sample][receptors]
            mat_dg_target = mats_d2c[sample][drugs]
            mat_pw = mats_pwa[sample]
            mat_tf = mats_tfa[sample]
            
            df_corr_drug.loc[drugs, sample] = mat_dg_target.corrwith(mat_dg, method='spearman')
            df_corr_gene.loc[genes, sample] = mat_gene.corrwith(mat_dg, method='spearman')
            df_corr_liga.loc[ligands, sample] = mat_ligand.corrwith(mat_dg, method='spearman')
            df_corr_rece.loc[receptors, sample] = mat_receptor.corrwith(mat_dg, method='spearman')
            df_corr_path.loc[pathway_list, sample] = mat_pw.corrwith(mat_dg, method='spearman')
            df_corr_tf.loc[tfs, sample] = mat_tf.corrwith(mat_dg, method='spearman')
    
    df_corr_drug.dropna(how='all', axis=1).dropna(how='all').to_csv(f'dataframes/correlation_ren/drug_with_drug/{drug}.csv')
    df_corr_gene.dropna(how='all', axis=1).dropna(how='all').to_csv(f'dataframes/correlation_ren/drug_with_gene/{drug}.csv')
    df_corr_liga.dropna(how='all', axis=1).dropna(how='all').to_csv(f'dataframes/correlation_ren/drug_with_ligand/{drug}.csv')
    df_corr_rece.dropna(how='all', axis=1).dropna(how='all').to_csv(f'dataframes/correlation_ren/drug_with_receptor/{drug}.csv')
    df_corr_path.dropna(how='all', axis=1).dropna(how='all').to_csv(f'dataframes/correlation_ren/drug_with_pathway/{drug}.csv')
    df_corr_tf.dropna(how='all', axis=1).dropna(how='all').to_csv(f'dataframes/correlation_ren/drug_with_tf/{drug}.csv')

## Correlation with genes

In [22]:
for gene in gene_list[:3]: ## remove [:3]
    df_corr_drug = pd.DataFrame(np.nan, index=drug_list, columns = sample_list)
    df_corr_gene = pd.DataFrame(np.nan, index=gene_list, columns = sample_list)
    df_corr_liga = pd.DataFrame(np.nan, index=ligand_list, columns = sample_list)
    df_corr_rece = pd.DataFrame(np.nan, index=receptor_list, columns = sample_list)
    df_corr_path = pd.DataFrame(np.nan, index=pathway_list, columns = sample_list)
    df_corr_tf = pd.DataFrame(np.nan, index=tf_list, columns = sample_list)
    
    for sample in sample_list:
        adata = adatas[sample]
        if gene in adata.var.index:
            genes = np.setdiff1d(np.intersect1d(adata.var_names, gene_list[:3]),
                                 [gene]).tolist() ## remove [:3]
            mat_gene = mat_neighbor[sample][gene]
            
            drugs = mats_d2c[sample].columns.to_list()
            ligands = np.setdiff1d(np.intersect1d(adata.var_names, ligand_list), [gene]).tolist() 
            receptors = np.setdiff1d(np.intersect1d(adata.var_names, receptor_list), [gene]).tolist() 
            tfs = mats_tfa[sample].columns.to_list()

            mat_gene_target = mat_neighbor[sample][genes]
            mat_ligand= mat_neighbor[sample][ligands]
            mat_receptor = mat_neighbor[sample][receptors]
            mat_dg = mats_d2c[sample]
            mat_tf = mats_tfa[sample]
            mat_pw = mats_pwa[sample]
            
            df_corr_drug.loc[drugs, sample] = mat_dg.corrwith(mat_gene, method='spearman')
            df_corr_gene.loc[genes, sample] = mat_gene_target.corrwith(mat_gene, method='spearman')
            df_corr_liga.loc[ligands, sample] = mat_ligand.corrwith(mat_gene, method='spearman')
            df_corr_rece.loc[receptors, sample] = mat_receptor.corrwith(mat_gene, method='spearman')
            df_corr_path.loc[pathway_list, sample] = mat_pw.corrwith(mat_gene, method='spearman')
            df_corr_tf.loc[tfs, sample] = mat_tf.corrwith(mat_gene, method='spearman')

    df_corr_drug.dropna(how='all', axis=1).dropna(how='all').to_csv(f'dataframes/correlation_ren/gene_with_drug/{gene}.csv')
    df_corr_gene.dropna(how='all', axis=1).dropna(how='all').to_csv(f'dataframes/correlation_ren/gene_with_gene/{gene}.csv')
    df_corr_liga.dropna(how='all', axis=1).dropna(how='all').to_csv(f'dataframes/correlation_ren/gene_with_ligand/{gene}.csv')
    df_corr_rece.dropna(how='all', axis=1).dropna(how='all').to_csv(f'dataframes/correlation_ren/gene_with_receptor/{gene}.csv')
    df_corr_path.dropna(how='all', axis=1).dropna(how='all').to_csv(f'dataframes/correlation_ren/gene_with_pathway/{gene}.csv')
    df_corr_tf.dropna(how='all', axis=1).dropna(how='all').to_csv(f'dataframes/correlation_ren/gene_with_tf/{gene}.csv')

## Correlation with ligands

In [23]:
for ligand in ligand_list[:3]: ## remove [:3]
    df_corr_drug = pd.DataFrame(np.nan, index=drug_list, columns = sample_list)
    df_corr_gene = pd.DataFrame(np.nan, index=gene_list, columns = sample_list) 
    df_corr_liga = pd.DataFrame(np.nan, index=ligand_list, columns = sample_list)
    df_corr_rece = pd.DataFrame(np.nan, index=receptor_list, columns = sample_list)
    df_corr_path = pd.DataFrame(np.nan, index=pathway_list, columns = sample_list)
    df_corr_tf = pd.DataFrame(np.nan, index=tf_list, columns = sample_list)
    
    for sample in sample_list:
        adata = adatas[sample]
        if ligand in adata.var.index:
            ligands = np.setdiff1d(np.intersect1d(adata.var_names, ligand_list), [ligand]).tolist() 
            mat_ligand= mat_neighbor[sample][ligand]
            
            drugs = mats_d2c[sample].columns.to_list()
            genes = np.intersect1d(adata.var_names, gene_list[:3]).tolist() ## remove [:3]
            receptors = np.intersect1d(adata.var_names, receptor_list).tolist()
            tfs = mats_tfa[sample].columns.to_list()
            
            mat_gene = mat_neighbor[sample][genes]
            mat_ligand_target = mat_neighbor[sample][ligands]
            mat_receptor = mat_neighbor[sample][receptors]
            mat_dg = mats_d2c[sample]
            mat_tf = mats_tfa[sample]
            mat_pw = mats_pwa[sample]

            df_corr_path.loc[pathway_list, sample] = mat_pw.corrwith(mat_ligand, method='spearman')
            df_corr_drug.loc[drugs, sample] = mat_dg.corrwith(mat_ligand, method='spearman')
            df_corr_gene.loc[genes, sample] = mat_gene.corrwith(mat_ligand, method='spearman')
            df_corr_liga.loc[ligands, sample] = mat_ligand_target.corrwith(mat_ligand, method='spearman')
            df_corr_rece.loc[receptors, sample] = mat_receptor.corrwith(mat_ligand, method='spearman')
            df_corr_tf.loc[tfs, sample] = mat_tf.corrwith(mat_ligand, method='spearman')
            
    df_corr_drug.dropna(how='all', axis=1).dropna(how='all').to_csv(f'dataframes/correlation_ren/ligand_with_drug/{ligand}.csv')
    df_corr_gene.dropna(how='all', axis=1).dropna(how='all').to_csv(f'dataframes/correlation_ren/ligand_with_gene/{ligand}.csv')
    df_corr_liga.dropna(how='all', axis=1).dropna(how='all').to_csv(f'dataframes/correlation_ren/ligand_with_ligand/{ligand}.csv')
    df_corr_rece.dropna(how='all', axis=1).dropna(how='all').to_csv(f'dataframes/correlation_ren/ligand_with_receptor/{ligand}.csv')
    df_corr_path.dropna(how='all', axis=1).dropna(how='all').to_csv(f'dataframes/correlation_ren/ligand_with_pathway/{ligand}.csv')
    df_corr_tf.dropna(how='all', axis=1).dropna(how='all').to_csv(f'dataframes/correlation_ren/ligand_with_tf/{ligand}.csv')

## Correlation with receptors

In [24]:
for receptor in receptor_list[:3]: ## remove [:3]
    df_corr_drug = pd.DataFrame(np.nan, index=drug_list, columns = sample_list)
    df_corr_gene = pd.DataFrame(np.nan, index=gene_list, columns = sample_list)
    df_corr_liga = pd.DataFrame(np.nan, index=ligand_list, columns = sample_list)
    df_corr_rece = pd.DataFrame(np.nan, index=receptor_list, columns = sample_list)
    df_corr_path = pd.DataFrame(np.nan, index=pathway_list, columns = sample_list)
    df_corr_tf = pd.DataFrame(np.nan, index=tf_list, columns = sample_list)
    
    for sample in sample_list:
        adata = adatas[sample]
        if receptor in adata.var.index:
            receptors = np.setdiff1d(np.intersect1d(adata.var_names, receptor_list), [receptor]).tolist() 
            mat_receptor = mat_neighbor[sample][receptor]
            
            drugs = mats_d2c[sample].columns.to_list()
            genes = np.intersect1d(adata.var_names, gene_list[:3]).tolist() ## remove [:3]
            ligands = np.intersect1d(adata.var_names, ligand_list).tolist()
            tfs = mats_tfa[sample].columns.to_list()
            
            mat_gene = mat_neighbor[sample][genes]
            mat_ligand= mat_neighbor[sample][ligands]
            mat_receptor_target = mat_neighbor[sample][receptors]
            mat_dg = mats_d2c[sample]
            mat_tf = mats_tfa[sample]
            mat_pw = mats_pwa[sample]
            
            df_corr_drug.loc[drugs, sample] = mat_dg.corrwith(mat_receptor, method='spearman')
            df_corr_gene.loc[genes, sample] = mat_gene.corrwith(mat_receptor, method='spearman')
            df_corr_liga.loc[ligands, sample] = mat_ligand.corrwith(mat_receptor, method='spearman')
            df_corr_rece.loc[receptors, sample] = mat_receptor_target.corrwith(mat_receptor, method='spearman')
            df_corr_path.loc[pathway_list, sample] = mat_pw.corrwith(mat_receptor, method='spearman')
            df_corr_tf.loc[tfs, sample] = mat_tf.corrwith(mat_receptor, method='spearman')

    df_corr_drug.dropna(how='all', axis=1).dropna(how='all').to_csv(f'dataframes/correlation_ren/receptor_with_drug/{receptor}.csv')
    df_corr_gene.dropna(how='all', axis=1).dropna(how='all').to_csv(f'dataframes/correlation_ren/receptor_with_gene/{receptor}.csv')
    df_corr_liga.dropna(how='all', axis=1).dropna(how='all').to_csv(f'dataframes/correlation_ren/receptor_with_ligand/{receptor}.csv')
    df_corr_rece.dropna(how='all', axis=1).dropna(how='all').to_csv(f'dataframes/correlation_ren/receptor_with_receptor/{receptor}.csv')
    df_corr_path.dropna(how='all', axis=1).dropna(how='all').to_csv(f'dataframes/correlation_ren/receptor_with_pathway/{receptor}.csv')
    df_corr_tf.dropna(how='all', axis=1).dropna(how='all').to_csv(f'dataframes/correlation_ren/receptor_with_tf/{receptor}.csv')

## Correlation with pathways

In [25]:
for pathway in pathway_list[:3]: ## remove [:3]
    df_corr_gene = pd.DataFrame(np.nan, index=gene_list, columns = sample_list)
    df_corr_drug = pd.DataFrame(np.nan, index=drug_list, columns = sample_list)
    df_corr_liga = pd.DataFrame(np.nan, index=ligand_list, columns = sample_list)
    df_corr_rece = pd.DataFrame(np.nan, index=receptor_list, columns = sample_list)
    df_corr_path = pd.DataFrame(np.nan, index=pathway_list, columns = sample_list)
    df_corr_tf = pd.DataFrame(np.nan, index=tf_list, columns = sample_list)
    
    for sample in sample_list:
        adata = adatas[sample]
        if pathway in mats_pwa[sample]:
            pathways = np.setdiff1d(mats_pwa[sample].columns, [pathway]).tolist()
            mat_pw = mats_pwa[sample][pathway]
            
            genes = np.intersect1d(adata.var_names, gene_list[:3]).tolist() ## remove [:3]
            drugs = mats_d2c[sample].columns.to_list()
            ligands = np.intersect1d(adata.var_names, ligand_list).tolist()
            receptors = np.intersect1d(adata.var_names, receptor_list).tolist()
            tfs = mats_tfa[sample].columns.to_list()
            
            mat_ligand = mat_neighbor[sample][ligands]
            mat_receptor = mat_neighbor[sample][receptors]
            mat_gene = mat_neighbor[sample][genes]
            mat_dg = mats_d2c[sample][drugs]
            mat_tf = mats_tfa[sample]
            mat_pw_target = mats_pwa[sample][pathways]

            df_corr_drug.loc[drugs, sample] = mat_dg.corrwith(mat_pw, method='spearman')
            df_corr_gene.loc[genes, sample] = mat_gene.corrwith(mat_pw, method='spearman')
            df_corr_liga.loc[ligands, sample] = mat_ligand.corrwith(mat_pw, method='spearman')
            df_corr_rece.loc[receptors, sample] = mat_receptor.corrwith(mat_pw, method='spearman')
            df_corr_path.loc[pathways, sample] = mat_pw_target.corrwith(mat_pw, method='spearman')
            df_corr_tf.loc[tfs, sample] = mat_tf.corrwith(mat_pw, method='spearman')
    
    df_corr_drug.dropna(how='all', axis=1).dropna(how='all').to_csv(f'dataframes/correlation_ren/pathway_with_drug/{pathway}.csv')
    df_corr_gene.dropna(how='all', axis=1).dropna(how='all').to_csv(f'dataframes/correlation_ren/pathway_with_gene/{pathway}.csv')
    df_corr_liga.dropna(how='all', axis=1).dropna(how='all').to_csv(f'dataframes/correlation_ren/pathway_with_ligand/{pathway}.csv')
    df_corr_rece.dropna(how='all', axis=1).dropna(how='all').to_csv(f'dataframes/correlation_ren/pathway_with_receptor/{pathway}.csv')
    df_corr_path.dropna(how='all', axis=1).dropna(how='all').to_csv(f'dataframes/correlation_ren/pathway_with_pathway/{pathway}.csv')
    df_corr_tf.dropna(how='all', axis=1).dropna(how='all').to_csv(f'dataframes/correlation_ren/pathway_with_tf/{pathway}.csv')

## Correlation with TFs

In [26]:
# for tf in tf_list: ## Change to this
for tf in ['SOX2', 'STAT3', 'HIF1A']:
    df_corr_drug = pd.DataFrame(np.nan, index=drug_list, columns = sample_list)
    df_corr_gene = pd.DataFrame(np.nan, index=gene_list, columns = sample_list)
    df_corr_liga = pd.DataFrame(np.nan, index=ligand_list, columns = sample_list)
    df_corr_rece = pd.DataFrame(np.nan, index=receptor_list, columns = sample_list)
    df_corr_path = pd.DataFrame(np.nan, index=pathway_list, columns = sample_list)
    df_corr_tf = pd.DataFrame(np.nan, index=tf_list, columns = sample_list)
    
    for sample in sample_list:
        adata = adatas[sample]
        if tf in mats_tfa[sample].columns:
            tfs = np.setdiff1d(mats_tfa[sample].columns, [tf]).tolist()
            mat_tf = mats_tfa[sample][tf]
            
            drugs = mats_d2c[sample].columns.to_list()
            genes = np.intersect1d(adata.var_names, gene_list).tolist()
            ligands = np.intersect1d(adata.var_names, ligand_list).tolist()
            receptors = np.intersect1d(adata.var_names, receptor_list).tolist()
            
            mat_ligand = mat_neighbor[sample][ligands]
            mat_receptor = mat_neighbor[sample][receptors]
            mat_gene = mat_neighbor[sample][genes]
            mat_dg = mats_d2c[sample][drugs]
            mat_pw = mats_pwa[sample]
            mat_tf_target = mats_tfa[sample][tfs]
            
            df_corr_drug.loc[drugs, sample] = mat_dg.corrwith(mat_tf, method='spearman')
            df_corr_gene.loc[genes, sample] = mat_gene.corrwith(mat_tf, method='spearman')
            df_corr_liga.loc[ligands, sample] = mat_ligand.corrwith(mat_tf, method='spearman')
            df_corr_rece.loc[receptors, sample] = mat_receptor.corrwith(mat_tf, method='spearman')
            df_corr_path.loc[pathway_list, sample] = mat_pw.corrwith(mat_tf, method='spearman')
            df_corr_tf.loc[tfs, sample] = mat_tf_target.corrwith(mat_tf, method='spearman')
            
    df_corr_drug.dropna(how='all', axis=1).dropna(how='all').to_csv(f'dataframes/correlation_ren/tf_with_drug/{tf}.csv')
    df_corr_gene.dropna(how='all', axis=1).dropna(how='all').to_csv(f'dataframes/correlation_ren/tf_with_gene/{tf}.csv')
    df_corr_liga.dropna(how='all', axis=1).dropna(how='all').to_csv(f'dataframes/correlation_ren/tf_with_ligand/{tf}.csv')
    df_corr_rece.dropna(how='all', axis=1).dropna(how='all').to_csv(f'dataframes/correlation_ren/tf_with_receptor/{tf}.csv')
    df_corr_path.dropna(how='all', axis=1).dropna(how='all').to_csv(f'dataframes/correlation_ren/tf_with_pathway/{tf}.csv')
    df_corr_tf.dropna(how='all', axis=1).dropna(how='all').to_csv(f'dataframes/correlation_ren/tf_with_tf/{tf}.csv')

In [27]:
marker_genes = ['VIM', 'CD44', 'IGFBP5', 
                'VEGFA', 'CAV1', 'LDHA', 'CD74', 'B2M',
                'SOX2', 'SOX10', 'SMAD4', 'FOS', 'JUN', 'ATF3']

for sample in sample_list:
    adata = adatas[sample]
    missing_genes = [gene for gene in marker_genes if gene not in adata.var_names]
    if missing_genes:
        print(f"{sample}: Missing {missing_genes}")
    else:
        print(f"{sample}: All marker genes present")

Ren2: Missing ['SOX10']
Ren3: All marker genes present
Ren51: All marker genes present
SNU16B: Missing ['LDHA']
SNU18A: Missing ['LDHA']
SNU18Afrozen: All marker genes present
SNU18B: Missing ['LDHA']
SNU21Afrozen: All marker genes present
SNU21B: Missing ['LDHA']
SNU21Dfrozen: All marker genes present
SNU24A: Missing ['LDHA']
SNU24B: Missing ['LDHA']
SNU25A: Missing ['LDHA']
SNU25B: Missing ['LDHA']
SNU27A: Missing ['LDHA']
SNU27B: Missing ['LDHA']
SNU43A: Missing ['LDHA']
SNU51A: Missing ['LDHA']
SNU51B: Missing ['LDHA']
